In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd
import numpy as np
import json
import shutil
from datetime import datetime

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Underwater-Image-Data-set-main"
)

DAY4_DIR = PROJECT_DIR / "Day_4_Preprocessing"

# Final frozen dataset version
V1_DIR = PROJECT_DIR / "Dataset_V1"

V1_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Dataset V1:", V1_DIR)

Mounted at /content/drive
Project: /content/drive/MyDrive/Underwater-Image-Data-set-main
Dataset V1: /content/drive/MyDrive/Underwater-Image-Data-set-main/Dataset_V1


In [2]:
def load_csv(filename):
    """
    Load a CSV if it exists.
    If it is empty, return an empty DataFrame.
    """

    path = PROJECT_DIR / filename

    if not path.exists():
        print(f"⚠️ Not found: {filename}")
        return pd.DataFrame()

    try:
        df = pd.read_csv(path)
        print(f"✓ {filename}: {df.shape}")
        return df

    except pd.errors.EmptyDataError:
        print(f"✓ {filename}: empty (0 entries)")
        return pd.DataFrame()


quality_summary = load_csv("dataset_quality_summary.csv")
incomplete_triplets = load_csv("incomplete_triplets.csv")
corrupt_images = load_csv("corrupt_images.csv")
resolution_problems = load_csv("resolution_problems.csv")
dark_images = load_csv("dark_images.csv")
blurry_images = load_csv("blurry_images.csv")
exact_duplicates = load_csv("exact_duplicates.csv")

✓ dataset_quality_summary.csv: (13, 2)
✓ incomplete_triplets.csv: (0, 5)
✓ corrupt_images.csv: empty (0 entries)
✓ resolution_problems.csv: empty (0 entries)
✓ dark_images.csv: (315, 7)
✓ blurry_images.csv: (306, 7)
✓ exact_duplicates.csv: (2591, 2)


In [3]:
import re

pattern = re.compile(
    r"img_(\d+)_(input|target|gen)",
    re.IGNORECASE
)

records = []

extensions = [
    "*.png",
    "*.jpg",
    "*.jpeg",
    "*.bmp",
    "*.tif",
    "*.tiff"
]

for ext in extensions:

    for path in PROJECT_DIR.rglob(ext):

        # Don't include files created for Dataset V1
        if V1_DIR in path.parents:
            continue

        # Don't include Day 4 outputs
        if DAY4_DIR in path.parents:
            continue

        match = pattern.fullmatch(path.stem)

        if match:

            image_id = int(match.group(1))
            role = match.group(2).lower()

            relative_parts = path.relative_to(PROJECT_DIR).parts

            source_archive = (
                relative_parts[0]
                if len(relative_parts) > 1
                else "unknown"
            )

            records.append({
                "image_id": image_id,
                "role": role,
                "file": str(path),
                "source_archive": source_archive
            })


images_df = pd.DataFrame(records)

print("Total role-labelled images:", len(images_df))
print("\nRole distribution:")
print(images_df["role"].value_counts())

Total role-labelled images: 942

Role distribution:
role
target    314
gen       314
input     314
Name: count, dtype: int64


In [4]:
triplets = (
    images_df
    .pivot_table(
        index=["source_archive", "image_id"],
        columns="role",
        values="file",
        aggfunc="first"
    )
    .reset_index()
)

# Keep only complete Input-Target-Generated triplets
triplets = triplets.dropna(
    subset=["input", "target", "gen"]
).reset_index(drop=True)

# Create globally unique sample identifier
triplets["sample_id"] = (
    triplets["source_archive"].astype(str)
    + "_"
    + triplets["image_id"].astype(str)
)

print("Complete triplets:", len(triplets))
print("Unique sample IDs:", triplets["sample_id"].nunique())

assert len(triplets) == triplets["sample_id"].nunique()

print("\n✓ Every triplet has a unique sample ID")

Complete triplets: 314
Unique sample IDs: 314

✓ Every triplet has a unique sample ID


In [5]:
train_path = DAY4_DIR / "train_split.csv"
val_path = DAY4_DIR / "validation_split.csv"
test_path = DAY4_DIR / "test_split.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Train: 219
Validation: 47
Test: 48
Total: 314


In [6]:
train_ids = set(train_df["sample_id"])
val_ids = set(val_df["sample_id"])
test_ids = set(test_df["sample_id"])

train_val = train_ids & val_ids
train_test = train_ids & test_ids
val_test = val_ids & test_ids

print("Train ∩ Validation:", train_val)
print("Train ∩ Test:", train_test)
print("Validation ∩ Test:", val_test)

assert len(train_val) == 0
assert len(train_test) == 0
assert len(val_test) == 0

print("✓ DATASET SPLIT IS LEAKAGE-SAFE")

Train ∩ Validation: set()
Train ∩ Test: set()
Validation ∩ Test: set()
✓ DATASET SPLIT IS LEAKAGE-SAFE


In [7]:
train_final = train_df.copy()
train_final["split"] = "train"

val_final = val_df.copy()
val_final["split"] = "validation"

test_final = test_df.copy()
test_final["split"] = "test"

final_splits = pd.concat(
    [train_final, val_final, test_final],
    ignore_index=True
)

print(final_splits["split"].value_counts())

final_splits.to_csv(
    V1_DIR / "Dataset_V1_splits.csv",
    index=False
)

print("\n✓ Saved Dataset V1 split table")

split
train         219
test           48
validation     47
Name: count, dtype: int64

✓ Saved Dataset V1 split table


In [8]:
import torchvision.transforms as transforms

IMAGE_SIZE = 224

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# Training preprocessing + mild augmentation
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=MEAN,
        std=STD
    )
])

# Validation/Test: NO augmentation
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=MEAN,
        std=STD
    )
])

print("Image size:", IMAGE_SIZE)
print("Normalization mean:", MEAN)
print("Normalization std:", STD)
print("Training augmentation: horizontal flip")
print("Validation/Test augmentation: NONE")

Image size: 224
Normalization mean: [0.485, 0.456, 0.406]
Normalization std: [0.229, 0.224, 0.225]
Training augmentation: horizontal flip
Validation/Test augmentation: NONE


In [9]:
config = {
    "dataset_version": "Dataset_V1",

    "dataset_structure": {
        "logical_sample": "source_archive + image_id",
        "image_roles": [
            "input",
            "target",
            "generated"
        ],
        "triplets_kept_together": True
    },

    "preprocessing": {
        "resize": "224x224",
        "normalization": "ImageNet",
        "mean": MEAN,
        "std": STD
    },

    "augmentation": {
        "training_only": True,
        "horizontal_flip": True,
        "probability": 0.5,
        "validation_augmentation": False,
        "test_augmentation": False
    },

    "splitting": {
        "train_percentage": 70,
        "validation_percentage": 15,
        "test_percentage": 15,
        "random_seed": 42,
        "split_unit": "logical triplet",
        "leakage_safe": True
    },

    "dataset_counts": {
        "complete_triplets": len(triplets),
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df)
    },

    "freeze_status": "FROZEN"
}

config_path = V1_DIR / "Dataset_V1_config.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("✓ Configuration saved:")
print(config_path)

✓ Configuration saved:
/content/drive/MyDrive/Underwater-Image-Data-set-main/Dataset_V1/Dataset_V1_config.json
